# Import libraries

In [ ]:
!pip install "protobuf==3.20.3" speechbrain

In [ ]:
import os
from datetime import datetime
from dataclasses import dataclass
from typing import Generator, Literal
import itertools
import random
import math
from dataclasses import dataclass

import pandas as pd
import soundfile as sf
from sklearn.model_selection import train_test_split
import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
import torch.nn as nn
from torch.nn.utils import spectral_norm
from torch.utils.data import IterableDataset, get_worker_info, DataLoader
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar, RichModelSummary, ModelSummary, TQDMProgressBar, DeviceStatsMonitor
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
from pytorch_lightning.tuner import Tuner
import IPython.display as ipd
from speechbrain.inference.vocoders import HIFIGAN

# Pretrain helpers

## Config

In [ ]:
@dataclass
class MixingAudioDatasetConfig:
    sample_rate: int
    segment_sec: float
    overlap: float
    min_snr: float
    max_snr: float
    skip_ratio: int


@dataclass
class AudioPreprocessorConfig:
    sample_rate: int
    n_fft: int
    window_length: int
    hop_length: int
    n_mels: int
    top_db: int
    mask_loss_threshold: float
    mask_loss_weight: float
    max_spec_shapes: tuple[int, int]
    spec_type: Literal["amplitude", "power"] = "amplitude"
    mel_scale: Literal["htk", "slaney"] = "htk"


@dataclass
class NormalizerConfig:
    min_db: float = -80.0
    max_db: float = 0.0
    scale_type: Literal["-1_1", "0_1"] = "-1_1"
    std: float = 1.0
    mean: float = 0.0


@dataclass
class AudioAugumentorConfig:
    time_mask_secs: float
    freq_mask_bins: int


@dataclass
class MelMelReGANTrainConfig:
    batch_size: int
    num_workers: int
    max_epochs: int
    learning_rate: float
    lambda_mag: float
    lambda_sc: float
    discriminator_train_freq: int
    label_smoothing: float
    warmup_epochs: int
    g_filters: int
    d_filters: int
    g_input_channels: int
    d_input_channels: int


@dataclass
class Pix2PixTrainConfig:
    batch_size: int
    num_workers: int
    max_epochs: int
    learning_rate: float
    lambda_recon: float


@dataclass
class LinearMelReGANTrainConfig:
    batch_size: int
    num_workers: int
    max_epochs: int
    learning_rate: float
    lambda_mag: float
    lambda_sc: float
    discriminator_train_freq: int
    g_filters: int
    d_filters: int
    d_input_channels: int


## DataFrames utils

In [ ]:
def merge_ears_filepaths_with_metadata(paths_df: pd.DataFrame, meta_df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Merge EARS file paths DataFrame with EARS person metadata DataFrame on 'person' column.

    Args:
        paths_df (pd.DataFrame): DataFrame containing file paths with 'person' column.
        meta_df (pd.DataFrame): DataFrame containing EARS person metadata with index as 'person'.

    Returns:
        pd.DataFrame: Merged DataFrame containing file paths and corresponding person metadata.
    """
    merged_df = pd.merge(paths_df, meta_df, left_on="person", right_index=True, how="inner")
    if verbose:
        print("Merged EARS Metadata Summary:")
        print(merged_df.head())
    return merged_df


def get_ears_personal_metadata(path: str) -> pd.DataFrame:
    """
    Load EARS person metadata from a JSON file.
    
    Args:
        path (str): Path to the JSON file containing EARS person metadata.

    Returns:
        pd.DataFrame: DataFrame containing the EARS person metadata.
    """
    meta_df = pd.read_json(path).T
    return meta_df


def preprocess_ears_metadata(path: str, fast: bool = True, verbose: bool = False) -> pd.DataFrame:
    """
    Preprocess EARS dataset metadata. Adds columns for length, style, emotion, freeform speech, and non-speech.

    Args:
        path (str): Path to the EARS dataset.
        fast (bool, optional): If True, skip length calculation for speed. Defaults to True.
        verbose (bool, optional): If True, print summary statistics. Defaults to False.

    Returns:
        pd.DataFrame: Preprocessed EARS metadata DataFrame.
    """
    df = _create_ears_paths(path)

    if not fast:
        df["length [s]"] = df["path"].apply(_count_len)

    df["style"] = df["file"].apply(_categorize_style)
    df["emotion"] = df["file"].apply(_categorize_emotion)
    df["is_freeform_speech"] = df["file"].apply(_categorize_freeform_speech)
    df["is_non_speech"] = df["file"].apply(_categorize_non_speech)

    df["style"] = df["style"].astype("category")
    df["emotion"] = df["emotion"].astype("category")

    if verbose:
        print("EARS Metadata Preprocessing Summary:")
        print(df.head())
    return df


def preprocess_wham_metadata(
        *,
        wham_data_cv: str,
        wham_data_tt: str,
        wham_data_tr: str,
        wham_noise_cv: str,
        wham_noise_tt: str,
        wham_noise_tr: str,
        wham_files_cv: str,
        wham_files_tt: str,
        wham_files_tr: str,
        fast: bool = True,
        verbose: bool = False
) -> pd.DataFrame:
    """
    Preprocess WHAM dataset metadata by merging data and noise CSV files and adding file paths.

    Args:
        wham_data_cv (str): Path to WHAM metadata cv CSV file.
        wham_data_tt (str): Path to WHAM metadata tt CSV file.
        wham_data_tr (str): Path to WHAM metadata tr CSV file.
        wham_noise_cv (str): Path to WHAM noise cv CSV file.
        wham_noise_tt (str): Path to WHAM noise tt CSV file.
        wham_noise_tr (str): Path to WHAM noise tr CSV file.
        wham_files_cv (str): Path to WHAM cv audio files directory.
        wham_files_tt (str): Path to WHAM tt audio files directory.
        wham_files_tr (str): Path to WHAM tr audio files directory.
        fast (bool, optional): If True, skip length calculation for speed. Defaults to True.
        verbose (bool, optional): If True, print summary statistics. Defaults to False.

    Returns:
        pd.DataFrame: Preprocessed WHAM metadata DataFrame.
    """
    wham_data_cv_df = pd.read_csv(wham_data_cv)
    wham_data_tt_df = pd.read_csv(wham_data_tt)
    wham_data_tr_df = pd.read_csv(wham_data_tr)

    wham_noise_cv_df = pd.read_csv(wham_noise_cv)
    wham_noise_tt_df = pd.read_csv(wham_noise_tt)
    wham_noise_tr_df = pd.read_csv(wham_noise_tr)

    wham_data_cv_df["path"] = wham_data_cv_df["utterance_id"].map(lambda f: os.path.join(wham_files_cv, f))
    wham_data_tt_df["path"] = wham_data_tt_df["utterance_id"].map(lambda f: os.path.join(wham_files_tt, f))
    wham_data_tr_df["path"] = wham_data_tr_df["utterance_id"].map(lambda f: os.path.join(wham_files_tr, f))

    wham_noise_df = pd.concat([wham_noise_tr_df, wham_noise_tt_df, wham_noise_cv_df], ignore_index=True)
    wham_data_df = pd.concat([wham_data_tr_df, wham_data_tt_df, wham_data_cv_df], ignore_index=True)
    wham_df = pd.merge(wham_noise_df, wham_data_df, on="utterance_id", how="inner")
    if not fast:
        wham_df["len"] = wham_df["path"].map(_count_len)

    if verbose:
        print("WHAM Metadata Preprocessing Summary:")
        print(wham_df.head())
    return wham_df


def prepare_for_training(df: pd.DataFrame, train_percentage: float, reduce_to: float | int | None, filter_to: dict[str, list[str]] | None = None, verbose: bool = False) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Prepare dataset by splitting into train, validate, and test sets. Also reduces the size of each set based on the reduce_to parameter.

    Args:
        df (pd.DataFrame): The input dataframe to split.
        train_percentage (float): The percentage of data to use for training set. Defaults to 0.8 (80%). Test and validation sets will each get half of the remaining data.
        reduce_to (float | int | None): If float in (0,1], reduces the dataset to that fraction. If int > 1, reduces the dataset to that many samples. If None, no reduction is applied.
        filter_to (dict[str, list[str]] | None): A dictionary where keys are column names and values are lists of acceptable values for those columns. If provided, filters the dataset before splitting. Defaults to None.
        verbose (bool): If True, prints the sizes of the resulting datasets.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: A tuple containing the train, validate, and test dataframes.
    """
    if filter_to is not None:
        df = _filter_dataset(df, filter_to)

    train_df, temp_df = train_test_split(df, train_size=train_percentage, random_state=42)
    validate_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    train_df = _reduce_dataset(train_df, reduce_to)
    validate_df = _reduce_dataset(validate_df, reduce_to)
    test_df = _reduce_dataset(test_df, reduce_to)
    if verbose:
        print(f"Train set size: {len(train_df)}")
        print(f"Validate set size: {len(validate_df)}")
        print(f"Test set size: {len(test_df)}")
    return train_df, validate_df, test_df


# ==============================================================================
#                       CATEGORY: PREPROCESS METADATA
#                 Helper functions for preprocessing metadata
# ==============================================================================


# ------------------------------------------------------------------------------
# EARS Dataset - Helper Functions
# ------------------------------------------------------------------------------
    
def _create_ears_paths(ears_dataset_path: str) -> pd.DataFrame:
    """
    Create a DataFrame with path, file, person information to all .wav files in the EARS dataset.

    Args:
        ears_dataset_path (str): Path to the EARS dataset.

    Returns:
        pd.DataFrame: DataFrame containing 'person', 'file', and 'path' columns for each .wav file.
    """
    rows = []
    max_person_index = 107
    for i in range(max_person_index):
        person = "p" + f"{i+1}".zfill(3) + "_resampled"
        person_dir = os.path.join(ears_dataset_path, person)
        
        if not os.path.exists(person_dir):
            raise ValueError(f"Path does not exists: {person_dir}")
        
        for file in os.listdir(person_dir):
            if file.endswith(".wav"):
                rows.append(
                    {
                        "person": person.replace("_resampled", ""),
                        "file": file,
                        "path": os.path.join(person_dir, file)
                    }
                )
        
    return pd.DataFrame(rows)

def _categorize_style(filename: str) -> str:
    """
    Categorize the speaking style based on given information in the filename.

    Args:
        filename (str): The filename to categorize.

    Returns:
        str: The categorized style from given set or "other" if none match.
    """
    styles = {"regular", "loud", "whisper", "fast", "slow", "highpitch", "lowpitch"}
    return next((s for s in styles if s in filename), "other")

def _categorize_emotion(filename: str) -> str:
    """
    Categorize the emotion based on given information in the filename.

    Args:
        filename (str): The filename to categorize.

    Returns:
        str: The categorized emotion from given set or "other" if none match.
    """
    emotions = {'adoration', 'fear', 'pain', 'realization', 'confusion',
       'cuteness', 'distress', 'guilt', 'amazement', 'contentment',
       'pride', 'desire', 'relief', 'disappointment', 'disgust',
       'embarassment', 'anger', 'interest', 'serenity', 'sadness',
       'amusement', 'extasy', 'neutral'}
    return next((e for e in emotions if e in filename), "other")
    
def _categorize_freeform_speech(filename: str) -> bool:
    """
    Categorize if the filename corresponds to clean long freeform speech.

    Args:
        filename (str): The filename to categorize.

    Returns:
        bool: True if the filename corresponds to freeform speech, False otherwise.
    """
    return True if "freeform_speech" in filename else False

def _categorize_non_speech(filename: str) -> bool:
    """
    Categorize if the filename corresponds to non-speech sounds.
    
    Args:
        filename (str): The filename to categorize.
        
    Returns:
        bool: True if the filename corresponds to non-speech sounds, False otherwise.
    """
    non_speech = {'interjection_greetings', 'vegetative_throat',
       'nonverbal_screaming', 'nonverbal_crying', 'vegetative_eating',
       'melodic_happy_birthday', 'vegetative_yawning',
       'nonverbal_laughter_open', 'nonverbal_cheering',
       'nonverbal_yelling', 'vegetative_coughing',
       'nonverbal_laughter_closed', 'interjection_agreement',
       'interjection_filler', 'vegetative_sneezing',
       'interjection_congratulations'}
    return any(ns in filename for ns in non_speech)


# ------------------------------------------------------------------------------
# WHAM Dataset - Helper Functions
# ------------------------------------------------------------------------------


# ------------------------------------------------------------------------------
# Common - Helper Functions
# ------------------------------------------------------------------------------

def _count_len(file: str) -> float:
    """
    Count the length of a .wav file in seconds.
    
    Args:
        file (str): Path to the .wav file.

    Returns:
        float: Length of the audio file in seconds.
    """
    data, samplerate = sf.read(file)
    return len(data) / samplerate


def _reduce_dataset(df: pd.DataFrame, reduce_param: float | int | None) -> pd.DataFrame:
    """
    Reduces the dataset based on the reduce_param.
    
    Args:
        df (pd.DataFrame): The input dataframe to reduce.
        reduce_param (float | int | None): If float in (0,1], reduces the dataset to that fraction. If int > 1, reduces the dataset to that many samples. If None, no reduction is applied.

    Returns:
        pd.DataFrame: The reduced dataframe.
    """
    if reduce_param is None:
        return df
    if isinstance(reduce_param, float) and 0 < reduce_param <= 1:
        return df.sample(frac=reduce_param, random_state=42)
    if isinstance(reduce_param, int) and reduce_param > 1:
        return df.sample(n=min(reduce_param, len(df)), random_state=42)
    raise ValueError("reduce_param musi być None, float w (0,1] albo int > 1")


def _filter_dataset(df: pd.DataFrame, filter: dict[str, list[str]]) -> pd.DataFrame:
    """
    Filters the dataset based on the provided filter criteria.

    Args:
        df (pd.DataFrame): The input dataframe to filter.
        filter (dict[str, list[str]]): A dictionary where keys are column names and values are lists of acceptable values for those columns.

    Returns:
        pd.DataFrame: The filtered dataframe.
    """
    for column, accepted_values in filter.items():
        df = df[df[column].isin(accepted_values)]
    return df

## Pipeline utils

In [ ]:
# ====================================================
# Abstract Classes  
# ====================================================

class Normalizer(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class Scaler(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError
    

class SpectrogramProcessor(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError
    

class Adjuster(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


# ====================================================
# Implementations  
# ====================================================

class AudioMixingDataset(IterableDataset):
    """Dataset for mixing clean audio with noise at random SNR levels."""
    def __init__(
            self,
            ears_df: pd.DataFrame,
            wham_df: pd.DataFrame,
            config: MixingAudioDatasetConfig,
            mode: Literal["train", "val", "test"],
            skip_ratio: int = 1
    ) -> None:
        self.ears_paths = ears_df["path"].tolist()
        self.wham_paths = wham_df["path"].tolist()
        self.c = config
        self.mode = mode
        self.sr = skip_ratio
        
        self.segment_samples = int(self.c.segment_sec * self.c.sample_rate)
        self.overlap_samples = int(self.c.overlap * self.c.sample_rate)
        self.step_samples = (self.segment_samples - self.overlap_samples) * self.sr
        
        if self.mode in ["val", "test"]:
            torch.manual_seed(42)
            self.fixed_snr_values = torch.empty(4000).uniform_(self.c.min_snr, self.c.max_snr)
            self.snr_index = 0
        else:
            self.fixed_snr_values = None
        
    def load_audio(self, path: str) -> torch.Tensor:
        """Load audio from a given file path."""
        waveform, sr = torchaudio.load(path)
        
        if sr != self.c.sample_rate:
            raise ValueError(f"Sample rate mismatch: expected {self.c.sample_rate}, got {sr}")
        
        return waveform.squeeze(0)
    
    def align_noise(self, noise: torch.Tensor, target_len: int) -> torch.Tensor:
        """Align noise to the target length by trimming or repeating."""
        noise_len = noise.shape[0]
        
        if noise_len == target_len:
            return noise
        
        if noise_len < target_len:
            repeats = (target_len + noise_len - 1) // noise_len
            return noise.repeat(repeats)[:target_len]
        else:
            start = torch.randint(0, noise_len - target_len + 1, (1,), dtype=torch.long).item()
            return noise[start:start + target_len]
    
    def cut_audio(self, audio: torch.Tensor, target_len: int) -> torch.Tensor:
        """Cut or pad audio to the target length."""
        audio_len = audio.shape[0]
        if audio_len > target_len:
            start = torch.randint(0, audio_len - target_len, (1,)).item()
            return audio[start:start + target_len]
        elif audio_len < target_len:
            padding = target_len - audio_len
            return torch.nn.functional.pad(audio, (0, padding))
        else:
            return audio
    
    def batch_add_noise(self, clean_windows: torch.Tensor, noise_windows: torch.Tensor) -> torch.Tensor:
        """Add noise to clean audio windows at specified SNR levels."""
        num_windows = clean_windows.shape[0]
        
        if self.mode == "train":
            snr_db = torch.empty(num_windows).uniform_(self.c.min_snr, self.c.max_snr)
        else:
            snr_db = torch.empty(num_windows)
            for i in range(num_windows):
                snr_db[i] = self.fixed_snr_values[self.snr_index % len(self.fixed_snr_values)]
                self.snr_index += 1
        
        return torch.vmap(F.add_noise)(clean_windows, noise_windows, snr_db)

    def __iter__(self) -> Generator[tuple[torch.Tensor, torch.Tensor], None, None]:
        """Iterator to yield mixed audio and clean audio pairs."""
        worker_info = get_worker_info()
        clean_paths = self.ears_paths[:]
        wham_paths = self.wham_paths[:]
        
        if worker_info is not None:
            per_worker = int(math.ceil(len(clean_paths) / float(worker_info.num_workers)))
            worker_id = worker_info.id
            iter_start = worker_id * per_worker
            iter_end = min(iter_start + per_worker, len(clean_paths))
            clean_paths = clean_paths[iter_start:iter_end]
        
        random.shuffle(wham_paths)
        noise_iter = itertools.cycle(wham_paths)
        
        if self.mode == "train":
            random.shuffle(clean_paths)
            
        for clean_path in clean_paths:
            clean_audio = self.load_audio(clean_path)

            if clean_audio.shape[0] < self.segment_samples:
                padding = self.segment_samples - clean_audio.shape[0]
                clean_audio = torch.nn.functional.pad(clean_audio, (0, padding))

            noise_path = next(noise_iter)
            noise_audio = self.load_audio(noise_path)
            
            noise_audio = self.align_noise(noise_audio, clean_audio.shape[0])
            
            clean_windows = clean_audio.unfold(0, self.segment_samples, self.step_samples)
            noise_windows = noise_audio.unfold(0, self.segment_samples, self.step_samples)

            mixed_windows = self.batch_add_noise(clean_windows, noise_windows)
            
            yield from zip(mixed_windows, clean_windows)


# ======= Spectrogram Processors =======

class AmplitudeSpectrogramProcessor(SpectrogramProcessor):
    """Compute amplitude spectrograms from audio signals."""
    def __init__(
            self,
            config: AudioPreprocessorConfig
    ) -> None:
        super().__init__()
        self.c = config
        self.spectrogram = T.Spectrogram(
            n_fft=self.c.n_fft,
            win_length=self.c.window_length,
            hop_length=self.c.hop_length
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute amplitude spectrogram and add channel dimension."""
        x = self.spectrogram(x)
        return x.unsqueeze(1)


class MelSpectrogramProcessor(SpectrogramProcessor):
    """Compute mel spectrograms from audio signals."""
    def __init__(
            self,
            config: AudioPreprocessorConfig
    ) -> None:
        super().__init__()
        self.c = config
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=self.c.sample_rate,
            n_fft=self.c.n_fft,
            win_length=self.c.window_length,
            hop_length=self.c.hop_length,
            n_mels=self.c.n_mels,
            f_min=0,
            f_max=self.c.sample_rate // 2,
            power=1.0 if self.c.spec_type == "amplitude" else 2.0,
            mel_scale=self.c.mel_scale,
            norm=self.c.mel_scale,
            center=True,
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute mel spectrogram and add channel dimension."""
        x = self.mel_spectrogram(x)
        return x.unsqueeze(1)


# ======= Scalers =======

class AudioToDBScaler(Scaler):
    """Convert amplitude or power spectrograms to decibel scale."""
    def __init__(self, audio_preprocessor_config: AudioPreprocessorConfig) -> None:
        super().__init__()
        self.top_db = audio_preprocessor_config.top_db
        self.stype = audio_preprocessor_config.spec_type
        self.amplitude_to_db = T.AmplitudeToDB(top_db=self.top_db, stype=self.stype)
    
    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Convert amplitude or power spectrogram to decibel scale."""
        return self.amplitude_to_db(spec)
    

class DBToLogScaler(Scaler):
    """
    Convert decibel spectrograms to the scale used by HiFi-GAN.
    HiFi-GAN uses a log10 scale with a specific top_db, so we need to convert from the standard amplitude-to-dB scale to the HiFi-GAN scale.

    The conversion is as follows:
    1. Convert from dB to linear scale using the formula: linear = 10^(dB / divider), where divider is 10 for amplitude spectrograms and 20 for power spectrograms.
    2. Apply natural logarithm and clamping as per HiFi-GAN implementation.
    """
    def __init__(self, audio_preprocessor_config: AudioPreprocessorConfig) -> None:
        super().__init__()
        self.cfg = audio_preprocessor_config
        self.factor = self._determine_factor(audio_preprocessor_config)

    def _determine_factor(self, cfg: AudioPreprocessorConfig) -> float:
        """Determine the divider factor based on the spectrogram type."""
        return 20.0 if cfg.spec_type == "amplitude" else 10.0

    def forward(self, spec_db: torch.Tensor) -> torch.Tensor:
        """Convert decibel spectrogram to log scale used by HiFi-GAN."""
        spec_linear = torch.pow(10.0, spec_db / self.factor)
        spec_log = torch.log(torch.clamp(spec_linear, min=1e-5))
        return spec_log
    

class AmplitudeToLog1pScaler(Scaler):
    """Convert amplitude spectrograms to log1p scale."""
    def __init__(self) -> None:
        super().__init__()

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Convert amplitude spectrogram to log1p scale."""
        spec = torch.log1p(spec)
        return spec


# ======= Normalizers =======

class StandardNormalizer(Normalizer):
    """Normalize spectrograms using standard normalization."""
    def __init__(self, config: NormalizerConfig) -> None:
        super().__init__()
        self.mean = config.mean
        self.std = config.std

    def normalize_standard(self, spec: torch.Tensor) -> torch.Tensor:
        """Normalize spectrogram using standard normalization."""
        return (spec - self.mean) / (self.std + 1e-12)

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Normalize spectrogram using standard normalization."""
        return self.normalize_standard(spec)


class MinMaxFixedNormalizer(Normalizer):
    """Normalize spectrograms using fixed min-max scaling."""
    def __init__(self, config: NormalizerConfig) -> None:
        super().__init__()
        self.min_db = config.min_db
        self.max_db = config.max_db
        self.db_range = config.max_db - config.min_db
        self.scale_type = config.scale_type

    def forward(self, spec_db: torch.Tensor) -> torch.Tensor:
        """Normalize spectrogram from decibel scale to normalized scale."""
        spec_db = torch.clamp(spec_db, min=self.min_db, max=self.max_db)
        spec_norm = (spec_db - self.min_db) / (self.db_range + 1e-12)
        if self.scale_type == "0_1":
            return spec_norm
        else:
            return spec_norm * 2.0 - 1.0

    def denormalize(self, spec_norm: torch.Tensor) -> torch.Tensor:
        """Denormalize spectrogram from normalized scale back to decibel scale."""
        if self.scale_type == "0_1":
            spec_0_1 = spec_norm
        else:
            spec_0_1 = (spec_norm + 1.0) / 2.0
        spec_db = spec_0_1 * self.db_range + self.min_db
        return spec_db


# ======= Augumentors =======

class AudioAugmentor(nn.Module):
    """Applies time and frequency masking to spectrograms."""
    def __init__(
            self,
            augumentor_config: AudioAugumentorConfig,
            audio_preprocessor_config: AudioPreprocessorConfig
    ) -> None:
        super().__init__()
        self.a_c = augumentor_config
        self.ap_c = audio_preprocessor_config
        self.time_mask_param = int(self.a_c.time_mask_secs * self.ap_c.sample_rate / self.ap_c.hop_length)
        self.freq_mask_param = self.a_c.freq_mask_bins
        self.time_mask = T.TimeMasking(time_mask_param=self.time_mask_param) if self.time_mask_param != 0 else None
        self.freq_mask = T.FrequencyMasking(freq_mask_param=self.freq_mask_param) if self.freq_mask_param is not None else None

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Apply time and frequency masking to the spectrogram."""
        if self.time_mask:
            spec = self.time_mask(spec)  
        if self.freq_mask:
            spec = self.freq_mask(spec) 
        return spec
    

# ====== Adjusters =======

class TrimAdjuster(Adjuster):
    """Adjuster that trims spectrograms to a target shape."""
    def __init__(self, audio_preprocessor_config: AudioPreprocessorConfig) -> None:
        super().__init__()
        self.target_shape = audio_preprocessor_config.max_spec_shapes

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Trim spectrogram to target shape."""
        spec = spec[:, :self.target_shape[0], :self.target_shape[1]]
        return spec


# ======= Pipelines =======

class DataPipeline(nn.Module):
    """Pipeline to process audio data through steps:
    - Preprocessing (e.g., spectrogram computation)
    - Augmentation (e.g., time/frequency masking)
    - Scaling (e.g., converting to dB or log scale)
    - Normalization (e.g., standard or min-max normalization)
    - Adjustment (e.g., trimming to target shape)
    """
    def __init__(
            self,
            preprocessor: SpectrogramProcessor = None,
            augmentor: AudioAugmentor = None,
            scale_converter: Scaler | None = None,
            scaler: Normalizer | None = None,
            adjuster: Adjuster | None = None
    ) -> None:
        super().__init__()
        self.preprocessor = preprocessor
        self.augmentor = augmentor
        self.scale_converter = scale_converter
        self.scaler = scaler
        self.adjuster = adjuster

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the training pipeline. Returns pair of input and target data"""
        if self.preprocessor:
            x = self.preprocessor(x)
            y = self.preprocessor(y)
        
        if self.augmentor:
            x = self.augmentor(x)

        if self.scale_converter:
            x = self.scale_converter(x)
            y = self.scale_converter(y)
        
        if self.scaler:
            x = self.scaler(x)        
            y = self.scaler(y)

        if self.adjuster:
            x = self.adjuster(x)
            y = self.adjuster(y)

        return x, y


# ======= DataModule =======

class AudioDataModule(pl.LightningDataModule):
    """DataModule for audio mixing dataset."""
    def __init__(
            self, 
            train_ds: AudioMixingDataset,
            val_ds: AudioMixingDataset,
            batch_size: int = 32, 
            num_workers: int = 4
        ) -> None:
        super().__init__()
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self) -> DataLoader:
        """Training dataloader."""
        return DataLoader(
            self.train_ds, 
            batch_size=self.batch_size, 
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False,
            prefetch_factor=4 if self.num_workers > 0 else None,
            drop_last=True,
            shuffle=False
        )

    def val_dataloader(self) -> DataLoader:
        """Validation dataloader."""
        return DataLoader(
            self.val_ds, 
            batch_size=self.batch_size, 
            num_workers=max(2, self.num_workers // 2),
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False,
            drop_last=False,
            shuffle=False
        )

## Callbacks

In [ ]:
class SpectrogramLogger(pl.Callback):
    """Callback to log spectrograms during validation."""
    def __init__(self, save_dir: str = "saved_spectrograms", num_samples: int = 10):
        super().__init__()
        self.save_dir = self._versionize_dir(save_dir)
        self.num_samples = num_samples
        os.makedirs(self.save_dir, exist_ok=True)

    def _versionize_dir(self, base_dir: str) -> str:
        """Create a versioned directory to avoid overwriting previous logs."""
        version = 0
        versioned_dir = os.path.join(base_dir, f"version_{version}")
        while os.path.exists(versioned_dir):
            version += 1
            versioned_dir = os.path.join(base_dir, f"version_{version}")
        return versioned_dir

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        """Log spectrograms at the end of the validation batch."""
        if batch_idx != 0 or outputs is None:
            return

        try:
            noisy_tensor = outputs["lossy_spec"]
            clean_tensor = outputs["clean_spec"]
            fake_tensor = outputs["fake_spec"]
        except KeyError:
            noisy_tensor = outputs.get("noisy")
            clean_tensor = outputs.get("clean")
            fake_tensor = outputs.get("fake")

        if noisy_tensor is None:
            print("SpectrogramLogger: Missing spectrograms in outputs, skipping logging.")
            return

        batch_size = noisy_tensor.shape[0]
        n = min(batch_size, self.num_samples)
        
        noisy_batch = noisy_tensor[:n].detach().cpu().numpy().squeeze(1)
        fake_batch = fake_tensor[:n].detach().cpu().numpy().squeeze(1)
        clean_batch = clean_tensor[:n].detach().cpu().numpy().squeeze(1)

        fig, axes = plt.subplots(n, 3, figsize=(12, 3 * n), squeeze=False)
        
        plot_kwargs = {'origin': 'lower', 'aspect': 'auto', 'cmap': 'magma', 'vmin': -1, 'vmax': 1, 'interpolation': 'nearest'}

        for i in range(n):
            im1 = axes[i, 0].imshow(noisy_batch[i], **plot_kwargs)
            axes[i, 0].set_ylabel(f"Sample {i}\nFreq (bins)", fontsize=10, fontweight='bold')
            
            im2 = axes[i, 1].imshow(fake_batch[i], **plot_kwargs)
            
            im3 = axes[i, 2].imshow(clean_batch[i], **plot_kwargs)

            cbar = fig.colorbar(im3, ax=axes[i, 2], fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=8)

            for col_idx, ax in enumerate(axes[i]):
                ax.tick_params(axis='both', which='major', labelsize=8)
                
                if i == n - 1:
                    ax.set_xlabel("Time (frames)", fontsize=10)
                else:
                    ax.set_xticklabels([])
                if col_idx > 0:
                    ax.set_yticklabels([])
            if i == 0:
                axes[i, 0].set_title("Input (Lossy/Noisy)", fontsize=14)
                axes[i, 1].set_title("Generated (Restored)", fontsize=14)
                axes[i, 2].set_title("Target (Clean)", fontsize=14)

        fig.suptitle(f'Validation Epoch {trainer.current_epoch}', fontsize=16, y=1.005)
        plt.tight_layout()

        if hasattr(trainer.logger, 'experiment'):
            try:
                trainer.logger.experiment.add_figure(f"Validation/Spectrograms_Batch", fig, global_step=trainer.global_step)
            except AttributeError:
                pass

        filename = f"epoch_{trainer.current_epoch:03d}_grid.png"
        save_path = os.path.join(self.save_dir, filename)
        fig.savefig(save_path, bbox_inches='tight')
        
        plt.close(fig)

## Utils

In [ ]:
def init_weights(m: nn.Module) -> None:
    """Initialize weights of convolutional and normalization layers."""
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)):
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)

## Mel Regenerative Generative Adversarial Network (MelReGAN)

### Layers

In [ ]:
class ConvEncoderBlock(nn.Module):
    """ Convolutional Encoder block: convolutional -> instance norm -> leakyrelu""" 
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int, 
            kernel_size: int = 4, 
            stride: int = 2, 
            use_norm: bool = True,
            use_activation: bool = True
        ) -> None:
        super().__init__()
        padding = (kernel_size - stride) // 2

        self.padding = nn.ReflectionPad2d(padding)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, bias=False if use_norm else True)
        self.norm = nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity()
        self.lrelu = nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the encoder block."""
        x = self.padding(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.lrelu(x)
        return x
    

class ConvDecoderBlock(nn.Module):
    """Convolutional Decoder block: upsample -> conv -> instance norm -> leakyrelu -> (optional dropout)"""
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int, 
            kernel_size: int = 3, 
            dropout: float = 0.0,
            use_norm: bool = True,
            use_activation: bool = True
        ) -> None:
        super().__init__()
        padding = (kernel_size - 1) // 2

        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        self.padding = nn.ReflectionPad2d(padding)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, bias=False if use_norm else True)
        self.norm = nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity()
        self.lrelu = nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the decoder block."""
        x = self.upsample(x)
        x = self.padding(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.lrelu(x)
        x = self.dropout(x)
        return x


class SpectralAttention(nn.Module):
    """Spectral Attention Mechanism focusing on frequency and time axes separately."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.channels = channels
        
        self.freq_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d((None, 1)),
            nn.Conv2d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid()
        )
        
        self.time_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, None)),
            nn.Conv2d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid()
        )
        
        self.alpha = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.ones(1))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the spectral attention mechanism."""
        freq_att = self.freq_attention(x)
        time_att = self.time_attention(x)
        out = x * (self.alpha * freq_att) * (self.beta * time_att)
        return out


class AttentionGate(nn.Module):
    """Attention Gate for skip connections in U-Net architecture."""
    def __init__(self, F_g: int, F_l: int, F_int: int) -> None:
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.InstanceNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.InstanceNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.InstanceNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the attention gate."""
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class DilatedResBlock(nn.Module):
    """Dilated Residual Block with dilation for larger receptive field.
    Schema: ReflectionPad -> Conv(dilated) -> InstanceNorm -> LeakyReLU -> Conv(1x1) -> InstanceNorm -> Add & Activation
    """
    def __init__(
            self, 
            channels: int,
            kernel_size: int = 3,
            dilation: int = 2,
            ) -> None:
        super().__init__()
        self.residual_block = nn.Sequential(
            nn.ReflectionPad2d(dilation),
            nn.Conv2d(channels, channels, kernel_size, dilation=dilation, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
        )
        self.activation = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the dilated residual block."""
        return self.activation(x + self.residual_block(x))


class ResidualEncoderBlock(nn.Module):
    """Residual Encoder block.
    Schema: residual block -> downsampling
    """
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int,
            kernel_size: int = 3,
            padding: int = 1,
            stride: int = 2,
            use_activation: bool = True,
            use_norm: bool = True
        ) -> None:
        super().__init__()
        self.res_block = nn.Sequential(
            nn.ReflectionPad2d(padding),
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.ReflectionPad2d(padding),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=0, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
        )
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity()
            )
        
        self.downsample = nn.Sequential(
            nn.ReflectionPad2d(padding),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, stride=stride, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
            nn.LeakyReLU(0.2, inplace=True)
        )

        self.final_activation = nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the residual encoder block."""
        residual = self.res_block(x)
        shortcut = self.shortcut(x)
        skip_out = self.final_activation(residual + shortcut)
        next_layer_in = self.downsample(skip_out)
        return skip_out, next_layer_in



class ResidualDecoderBlock(nn.Module):
    """Decoder block with Residual Connection:
    Schema: Upsample -> Reduce Channels -> Residual Block -> Add & Activation -> (Optional Dropout)
    """
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int, 
            kernel_size: int = 3,
            padding: int = 1,
            stride: int = 1,
            dropout: float = 0.0,
            use_norm: bool = True,
            use_activation: bool = True,
            upsample: bool = True
        ) -> None:
        super().__init__()
        
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest') if upsample else nn.Identity()
        
        self.reduce = nn.Sequential(
            nn.ReflectionPad2d(padding),
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
            nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()
        )
        
        self.res_block = nn.Sequential(
            nn.ReflectionPad2d(padding),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
            nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity(),
            
            nn.ReflectionPad2d(padding),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, bias=False),
            nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity(),
        )
        
        self.final_activation = nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()
        
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the residual decoder block."""
        x = self.upsample(x)
        x = self.reduce(x)
        residual = self.res_block(x)
        x = self.final_activation(x + residual)
        x = self.dropout(x)
        return x


class PatchBlock(nn.Module):
    """PatchGAN Discriminator Block: Conv -> InstanceNorm -> LeakyReLU"""
    def __init__(
            self, 
            in_channels: int, 
            out_channels: int, 
            kernel_size: tuple[int, int] = (8, 2),
            stride: int = 2,
            padding: tuple[int, int] = (3, 1),
            use_norm: bool = True,
            use_activation: bool = True
        ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=not use_norm)
        self.norm = nn.InstanceNorm2d(out_channels, affine=True) if use_norm else nn.Identity()
        self.activation = nn.LeakyReLU(0.2, inplace=True) if use_activation else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the PatchGAN discriminator block."""
        x = self.conv(x)
        x = self.norm(x)
        x = self.activation(x)
        return x

### Models

In [ ]:
class MelReGANGenerator(nn.Module):
    """U-Net based Generator with Residual Blocks and Attention Gates."""
    def __init__(self, start_filters: int = 64) -> None:
        super().__init__()
        nf = start_filters
        
        self.encoder1 = ResidualEncoderBlock(1, nf)      
        self.encoder2 = ResidualEncoderBlock(nf, nf*2)                   
        self.encoder3 = ResidualEncoderBlock(nf*2, nf*4)                 
        self.encoder4 = ResidualEncoderBlock(nf*4, nf*8) 
        
        self.bottleneck = nn.Sequential(
            DilatedResBlock(nf*8, dilation=2),
            # SpectralAttention(nf*8),
            DilatedResBlock(nf*8, dilation=4),
            # SpectralAttention(nf*8),
            DilatedResBlock(nf*8, dilation=4),
            SpectralAttention(nf*8)
        )
        
        self.gate1 = AttentionGate(F_g=nf*4, F_l=nf*4, F_int=nf*2) 
        self.gate2 = AttentionGate(F_g=nf*2, F_l=nf*2, F_int=nf)   
        self.gate3 = AttentionGate(F_g=nf,   F_l=nf,   F_int=nf//2)
        
        self.decoder1 = ResidualDecoderBlock(nf*8, nf*4, dropout=0.5)    
        self.decoder2 = ResidualDecoderBlock(nf*4*2, nf*2, dropout=0.5)  
        self.decoder3 = ResidualDecoderBlock(nf*2*2, nf, dropout=0.0)
        self.final = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(nf*2, 1, kernel_size=3, stride=1, padding=0),
            nn.Tanh()
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the MelReGAN Generator."""
        e1_skip, e1_next = self.encoder1(x)      
        e2_skip, e2_next = self.encoder2(e1_next)     
        e3_skip, e3_next = self.encoder3(e2_next)     
        e4_skip, _       = self.encoder4(e3_next)     
        
        x_bottleneck = self.bottleneck(e4_skip)
        
        d1 = self.decoder1(x_bottleneck)        
        e3_gated = self.gate1(d1, e3_skip)      
        d1 = torch.cat([d1, e3_gated], dim=1)   
        
        d2 = self.decoder2(d1)                  
        e2_gated = self.gate2(d2, e2_skip)      
        d2 = torch.cat([d2, e2_gated], dim=1)   

        d3 = self.decoder3(d2)                  
        e1_gated = self.gate3(d3, e1_skip)      
        d3 = torch.cat([d3, e1_gated], dim=1)   
        
        output = self.final(d3)
        return output


class MelReGANDiscriminator(nn.Module):
    """PatchGAN Discriminator."""
    def __init__(
            self, 
            in_channels: int = 2, 
            start_filters: int = 64
        ) -> None:
        super().__init__()
        nf = start_filters

        # Layer 1: Stride 2 (Downsample) -> Output: 40x64
        self.layer_1 = PatchBlock(in_channels, nf, kernel_size=(4,4), stride=2)
        
        # Layer 2: Stride 1 (Zachowaj rozmiar) -> Output: 40x64
        self.layer_2 = PatchBlock(nf, nf * 2, kernel_size=(4,4), stride=2)
        
        # # Layer 3: Stride 1 -> Output: 40x64
        # self.layer_3 = PatchBlock(nf * 2, nf * 4, kernel_size=(4,4), stride=1)
        
        self.final = nn.Conv2d(nf * 2, 1, kernel_size=(4,4), stride=1, padding=1)

    def forward(self, x: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        d_in = torch.cat([x, condition], dim=1)
        d1 = self.layer_1(d_in)
        d2 = self.layer_2(d1)
        # d3 = self.layer_3(d2)
        output = self.final(d2)
        return output

### Losses

In [ ]:
class MaskedSpectralLoss(nn.Module):
    """Computes Masked Magnitude Loss and Spectral Convergence Loss."""
    def __init__(self, scaler: nn.Module, cfg: AudioPreprocessorConfig) -> None:
        super().__init__()
        self.scaler = scaler
        self.cfg = cfg
        self.factor = 20.0 if self.cfg.spec_type == "amplitude" else 10.0
        self.threshold = self.cfg.mask_loss_threshold
        self.mask_weight = self.cfg.mask_loss_weight
        self.log_conversion = torch.log(torch.tensor(10.0)) / self.factor


    def forward(self, pred_norm: torch.Tensor, target_norm: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Compute the Masked Magnitude Loss and Spectral Convergence Loss."""
        threshold = torch.max(target_norm.reshape(target_norm.size(0), -1), dim=1)[0] * self.threshold
        threshold = threshold.reshape(-1, 1, 1, 1)
        
        mask = torch.ones_like(target_norm)
        mask[target_norm < threshold] = self.mask_weight
        
        l1_diff = torch.abs(pred_norm - target_norm)
        l_mag = torch.mean(l1_diff * mask) 

        pred_db = self.scaler.denormalize(pred_norm)
        target_db = self.scaler.denormalize(target_norm)
        
        pred_lin = torch.exp(pred_db * self.log_conversion)
        target_lin = torch.exp(target_db * self.log_conversion)
        
        diff = target_lin - pred_lin
        numerator = torch.sqrt(torch.sum(diff * diff) + 1e-12)
        denominator = torch.sqrt(torch.sum(target_lin * target_lin) + 1e-12)
        l_sc = numerator / denominator
        
        return l_mag, l_sc

### Strategies

In [ ]:
class MelReGAN(pl.LightningModule):
    """Lightning Module for training Mel Regenerative Generative Adversarial Network (MelReGAN)."""
    def __init__(
            self, 
            generator: MelReGANGenerator, 
            discriminator: MelReGANDiscriminator,
            pipeline: nn.Module,
            scaler: nn.Module,
            audio_cfg: AudioPreprocessorConfig,
            lr: float = 0.0002,
            lambda_mag: float = 250.0, 
            lambda_sc: float = 250.0,
            discriminator_train_freq: int = 1,
            label_smoothing: float = 0.9,
            warmup_epochs: int = 5
        ) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=["generator", "discriminator", "pipeline", "scaler"])
        
        self.generator = generator
        self.discriminator = discriminator
        self.pipeline = pipeline
        self.scaler = scaler
        self.audio_cfg = audio_cfg

        self.ls = label_smoothing
        
        self.spectral_losses = MaskedSpectralLoss(scaler=scaler, cfg=audio_cfg)    
        self.adversarial_loss = nn.MSELoss()
        
        self.automatic_optimization = False
        self.gradient_clip_val = 1.0
        self.example_input_array = torch.randn(32, 1, 80, 128)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the generator and discriminator."""
        fake_spec = self.generator(x)
        disc_out = self.discriminator(fake_spec, x)
        return fake_spec, disc_out

    def configure_optimizers(self) -> tuple[list[torch.optim.Optimizer], list[torch.optim.lr_scheduler._LRScheduler]]:
       """Configure optimizers and learning rate schedulers."""
       opt_g = torch.optim.Adam(self.generator.parameters(), lr=self.hparams.lr, betas=(0.5, 0.999))
       opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=self.hparams.lr, betas=(0.5, 0.999))
       
       scheduler_g = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_g, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6)
       scheduler_d = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_d, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6)
        
       return [opt_g, opt_d], [scheduler_g, scheduler_d]

    def training_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> None:
        """Training step for MelReGAN model."""
        g_opt, d_opt = self.optimizers()
        mixed_audio, clean_audio = batch
        
        real_lossy_spec, real_clean_spec = self.pipeline(mixed_audio, clean_audio)
        
        is_warmup = self.current_epoch < self.hparams.warmup_epochs
        train_d = (batch_idx % self.hparams.discriminator_train_freq == 0) and (not is_warmup)
        
        loss_d_total = None 

        if train_d:
            with torch.no_grad():
                fake_clean_spec_detached = self.generator(real_lossy_spec)
            
            # REAL CLEAN
            pred_real = self.discriminator(real_clean_spec, real_lossy_spec)
            loss_d_real = self.adversarial_loss(pred_real, torch.ones_like(pred_real) * self.ls)
            
            # FAKE CLEAN
            pred_fake_d = self.discriminator(fake_clean_spec_detached, real_lossy_spec)
            loss_d_fake = self.adversarial_loss(pred_fake_d, torch.zeros_like(pred_fake_d))
            
            # LOSS D
            loss_d_total = (loss_d_real + loss_d_fake) * 0.5
            
            # BACKWARD AND OPTIMIZE D
            d_opt.zero_grad()
            self.manual_backward(loss_d_total)
            torch.nn.utils.clip_grad_norm_(self.discriminator.parameters(), self.gradient_clip_val)
            d_opt.step()
        
        # ----------------------------------------------------------------------
        # TRAIN GENERATOR
        # ----------------------------------------------------------------------
        fake_clean_spec = self.generator(real_lossy_spec)
        
        if is_warmup:
            loss_gan = torch.tensor(0.0, device=self.device)
        else:
            pred_fake_g = self.discriminator(fake_clean_spec, real_lossy_spec)
            loss_gan = self.adversarial_loss(pred_fake_g, torch.ones_like(pred_fake_g))
            
        # SPECTRAL LOSSES
        loss_mag, loss_sc = self.spectral_losses(fake_clean_spec, real_clean_spec)
        
        # TOTAL GENERATOR LOSS
        total_gen_loss = loss_gan + \
                         (self.hparams.lambda_mag * loss_mag) + \
                         (self.hparams.lambda_sc * loss_sc)
                         
        # BACKWARD AND OPTIMIZE G
        g_opt.zero_grad()
        self.manual_backward(total_gen_loss)
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), self.gradient_clip_val)
        g_opt.step()
        
        # LOG METRICS
        log_metrics = {
            "g_total": total_gen_loss.detach(), 
            "g_gan": loss_gan.detach(), 
            "g_mag": loss_mag.detach(), 
            "g_sc": loss_sc.detach(),
        }
        
        if loss_d_total is not None:
            log_metrics["d_loss"] = loss_d_total.detach()

        self.log_dict(log_metrics, prog_bar=True, on_step=False, on_epoch=True)
    
    def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> dict:
        """Validation step for MelReGAN model."""
        mixed_audio, clean_audio = batch
        real_lossy_spec, real_clean_spec = self.pipeline(mixed_audio, clean_audio)
        fake_clean_spec = self.generator(real_lossy_spec)

        loss_mag, loss_sc = self.spectral_losses(fake_clean_spec, real_clean_spec)
        val_loss = loss_mag + loss_sc
        self.log("val_loss", val_loss, prog_bar=True, on_step=False, on_epoch=True)
        return {
            "lossy_spec": real_lossy_spec,
            "clean_spec": real_clean_spec,
            "fake_spec": fake_clean_spec,
            "loss": val_loss
        }
    
    def on_validation_epoch_end(self) -> None:
       """Called at the end of the validation epoch to step LR schedulers."""
       schedulers = self.lr_schedulers()
       val_loss = self.trainer.callback_metrics.get("val_loss")
       if val_loss is not None and schedulers:
           if isinstance(schedulers, list):
               for scheduler in schedulers:
                   scheduler.step(val_loss)
           else:
               schedulers.step(val_loss)

### Train utils

In [ ]:
def create_configs() -> dict[str, dataclass]:
    """Create and return configuration dataclasses for the training pipeline."""
    return {
        "mixing_audio_cfg": MixingAudioDatasetConfig(
            sample_rate=16000,
            segment_sec=2.04,
            overlap=0.0,
            min_snr=-2.5,
            max_snr=17.5,
            skip_ratio=2
        ),
        "audio_preprocessor_cfg": AudioPreprocessorConfig(
            sample_rate=16000,
            n_fft=1024,
            window_length=1024,
            hop_length=256,
            n_mels=80,
            top_db=80,
            mask_loss_threshold=0.1,
            mask_loss_weight=0.5,
            spec_type="amplitude",
            mel_scale="slaney",
            max_spec_shapes=(80, 128)
        ),
        "normalizer_cfg": NormalizerConfig(
            min_db=-80.0,
            max_db=0.0,
            scale_type="-1_1",
            std=1.0,
            mean=0.0
        ),
        "audio_augumentor_cfg": AudioAugumentorConfig(
            time_mask_secs=0.12,
            freq_mask_bins=None
        ),
        "train_cfg": MelMelReGANTrainConfig(
            batch_size=32,
            num_workers=4,
            max_epochs=200,
            learning_rate=0.0002,
            lambda_mag=45.0,
            lambda_sc=25.0,
            discriminator_train_freq=2,
            label_smoothing=0.9,
            warmup_epochs=5,
            g_filters=32,
            d_filters=32,
            g_input_channels=1,
            d_input_channels=2
        )
    }


def split_dataframes(
        ears_df: pd.DataFrame, 
        wham_df: pd.DataFrame, 
        train_percentage: float = 0.8, 
        reduced_size: int | float | None = None
    ) -> dict[str, pd.DataFrame]:
    """Split EARS and WHAM dataframes into training, validation, and test sets."""
    train_ears_df, val_ears_df, test_ears_df = prepare_for_training(
        ears_df, train_percentage=train_percentage, reduce_to=reduced_size, verbose=True
    )
    train_wham_df, val_wham_df, test_wham_df = prepare_for_training(
        wham_df, train_percentage=train_percentage, reduce_to=reduced_size, verbose=True
    )
    return {
        "train_ears_df": train_ears_df,
        "val_ears_df": val_ears_df,
        "test_ears_df": test_ears_df,
        "train_wham_df": train_wham_df,
        "val_wham_df": val_wham_df,
        "test_wham_df": test_wham_df
    }


def create_dataframes(shared_path: str, kaggle: bool = False) -> dict[str, pd.DataFrame]:
    """Load and preprocess metadata for EARS and WHAM datasets."""
    if kaggle:
        COMMON_PATH = os.path.join(shared_path, "speech-enhancement")
        EARS_DATASET = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset", "speaker_statistics.json")
        EARS_FILES = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset", "ears_dataset_resampled")
        WHAM_DATA_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_tt.csv")
        WHAM_DATA_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_tr.csv")
        WHAM_DATA_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_cv.csv")
        WHAM_DATA_NOISE_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_tt.csv")
        WHAM_DATA_NOISE_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_tr.csv")
        WHAM_DATA_NOISE_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_cv.csv")
        WHAM_FILES_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_tt")
        WHAM_FILES_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_cv")
        WHAM_FILES_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_tr")
    else:
        COMMON_PATH = shared_path
        EARS_DATASET = os.path.join(COMMON_PATH, "ears_dataset", "speaker_statistics.json")
        EARS_FILES = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset_resampled")
        WHAM_DATA_TT = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_tt.csv")
        WHAM_DATA_TR = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_tr.csv")
        WHAM_DATA_CV = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_cv.csv")
        WHAM_DATA_NOISE_TT = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_tt.csv")
        WHAM_DATA_NOISE_TR = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_tr.csv")
        WHAM_DATA_NOISE_CV = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_cv.csv")
        WHAM_FILES_TT = os.path.join(COMMON_PATH, "wham_noise", "resampled_tt")
        WHAM_FILES_CV = os.path.join(COMMON_PATH, "wham_noise", "resampled_cv")
        WHAM_FILES_TR = os.path.join(COMMON_PATH, "wham_noise", "resampled_tr")

    personal_metadata_df = get_ears_personal_metadata(EARS_DATASET)
    ears_metadata_df = preprocess_ears_metadata(EARS_FILES, verbose=False)
    wham_df = preprocess_wham_metadata(
        wham_data_cv=WHAM_DATA_CV,
        wham_data_tr=WHAM_DATA_TR,
        wham_data_tt=WHAM_DATA_TT,
        wham_files_cv=WHAM_FILES_CV,
        wham_files_tr=WHAM_FILES_TR,
        wham_files_tt=WHAM_FILES_TT,
        wham_noise_cv=WHAM_DATA_NOISE_CV,
        wham_noise_tr=WHAM_DATA_NOISE_TR,
        wham_noise_tt=WHAM_DATA_NOISE_TT,
        verbose=False
    )
    ears_df = merge_ears_filepaths_with_metadata(ears_metadata_df, personal_metadata_df)
    return {"wham_df": wham_df, "ears_df": ears_df}


def create_dataset(
        ears_df: pd.DataFrame, 
        wham_df: pd.DataFrame, 
        mixing_audio_cfg: MixingAudioDatasetConfig,
        *,
        train: bool = True, 
        skip_ratio: int = 2
    ) -> AudioMixingDataset:
    """Create an AudioMixingDataset for training or validation."""
    ds = AudioMixingDataset(
        ears_df=ears_df,
        wham_df=wham_df,
        config=mixing_audio_cfg,
        mode="train" if train else "val",
        skip_ratio=skip_ratio
    )
    return ds


def create_scaler(cfg: NormalizerConfig) -> MinMaxFixedNormalizer:
    """Create scaler"""
    return MinMaxFixedNormalizer(cfg)


def create_pipeline(
        audio_preprocessor_cfg: AudioPreprocessorConfig, 
        audio_augmentor_cfg: AudioAugumentorConfig, 
        normalizer_cfg: NormalizerConfig
    ) -> DataPipeline:
    """Create the training pipeline with preprocessor, augmentor, scaler, and adjuster."""
    preprocessor = MelSpectrogramProcessor(config=audio_preprocessor_cfg)
    scale_converter = AudioToDBScaler(audio_preprocessor_cfg)
    augmentor = AudioAugmentor(
        audio_preprocessor_config=audio_preprocessor_cfg, 
        augumentor_config=audio_augmentor_cfg
    )
    adjuster = TrimAdjuster(audio_preprocessor_cfg)
    scaler = create_scaler(normalizer_cfg)

    pipeline = DataPipeline(
        preprocessor=preprocessor,
        augmentor=augmentor,
        scaler=scaler,
        adjuster=adjuster,
        scale_converter=scale_converter
    )
    return pipeline


def create_data_module(
        train_dataset: AudioMixingDataset, 
        val_dataset: AudioMixingDataset, 
        batch_size: int, 
        num_workers: int
    ) -> AudioDataModule:
    """Create a PyTorch Lightning DataModule for training."""
    data_module = AudioDataModule(
        train_ds=train_dataset, 
        val_ds=val_dataset, 
        batch_size=batch_size,
        num_workers=num_workers 
    )
    return data_module


def create_strategy(
        generator: MelReGANGenerator, 
        discriminator: MelReGANDiscriminator, 
        pipeline: DataPipeline, 
        scaler: Normalizer,
        audio_cfg: AudioPreprocessorConfig,
        cfg: MelMelReGANTrainConfig
    ) -> MelReGAN:
    """Create the MelReGAN model for training."""
    model = MelReGAN(
        generator=generator,
        discriminator=discriminator,
        pipeline=pipeline,
        scaler=scaler,
        audio_cfg=audio_cfg,
        lr=cfg.learning_rate,
        lambda_mag=cfg.lambda_mag,
        lambda_sc=cfg.lambda_sc,
        discriminator_train_freq=cfg.discriminator_train_freq
    )
    return model


def create_callbacks() -> list:
    """Create a list of callbacks for training."""
    checkpoint_callback = ModelCheckpoint(
        dirpath="checkpoints",
        filename="mel_MelReGAN-{epoch:02d}-{val_loss:.4f}",
        save_top_k=3,
        monitor="val_loss",
        verbose=True,
        save_weights_only=True,
        mode="min"
    )

    early_stopping_callback = EarlyStopping(
        monitor="val_loss",
        patience=40,
        verbose=True,
        mode="min"
    )

    image_callback = SpectrogramLogger()
    progress_bar_callback = RichProgressBar(leave=True)
    model_summary_callback = RichModelSummary(max_depth=2)
    dev_stats_callback = DeviceStatsMonitor()

    return [checkpoint_callback, early_stopping_callback, image_callback, progress_bar_callback, model_summary_callback, dev_stats_callback]


def create_loggers() -> tuple[TensorBoardLogger, CSVLogger]:
    """Create TensorBoard and CSV loggers."""
    logger = TensorBoardLogger(save_dir="tb_logs", name="mel_MelReGAN")
    csv_logger = CSVLogger(save_dir="csv_logs", name="mel_MelReGAN")
    return logger, csv_logger

def create_trainer(
        train_size: int,
        train_percentage: float,
        max_epochs: int,
        cfg: MelMelReGANTrainConfig,
        loggers: list,
        callbacks: list,
):
    limit_train_batches = train_size // cfg.batch_size
    limit_val_batches = (1 - train_percentage) * train_size // cfg.batch_size

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=callbacks,
        logger=loggers,
        limit_train_batches=limit_train_batches,
        limit_val_batches=limit_val_batches,
        accelerator="auto",
        devices="auto",
        benchmark=True,
        deterministic=False,
        log_every_n_steps=50,
    )
    return trainer



def train(train_size: int = 10_000, train_percentage: float = 0.8) -> tuple[pl.Trainer, MelReGAN, AudioDataModule]:
    """Main function to set up and start training the MelReGAN model."""
    configs = create_configs()
    dataframes = create_dataframes(shared_path="/kaggle/input", kaggle=True)
    split_dfs = split_dataframes(dataframes["ears_df"], dataframes["wham_df"], train_percentage=train_percentage, reduced_size=None)
    
    train_dataset = create_dataset(
        ears_df=split_dfs["train_ears_df"],
        wham_df=split_dfs["train_wham_df"],
        mixing_audio_cfg=configs["mixing_audio_cfg"],
        train=True,
        skip_ratio=configs["mixing_audio_cfg"].skip_ratio
    )

    val_dataset = create_dataset(
        ears_df=split_dfs["val_ears_df"],
        wham_df=split_dfs["val_wham_df"],
        mixing_audio_cfg=configs["mixing_audio_cfg"],
        train=False,
        skip_ratio=configs["mixing_audio_cfg"].skip_ratio
    )

    pipeline = create_pipeline(
        audio_preprocessor_cfg=configs["audio_preprocessor_cfg"], 
        audio_augmentor_cfg=configs["audio_augumentor_cfg"], 
        normalizer_cfg=configs["normalizer_cfg"]
    )

    data_module = create_data_module(
        train_dataset=train_dataset, 
        val_dataset=val_dataset, 
        batch_size=configs["train_cfg"].batch_size, 
        num_workers=configs["train_cfg"].num_workers
    )

    generator = MelReGANGenerator(start_filters=configs["train_cfg"].g_filters)
    discriminator = MelReGANDiscriminator(
        in_channels=configs["train_cfg"].d_input_channels, 
        start_filters=configs["train_cfg"].d_filters
    )

    generator.apply(init_weights)
    discriminator.apply(init_weights)

    model = create_strategy(
        generator=generator,
        discriminator=discriminator,
        pipeline=pipeline,
        scaler=create_scaler(configs["normalizer_cfg"]),
        audio_cfg=configs["audio_preprocessor_cfg"],
        cfg=configs["train_cfg"]
    )

    callbacks = create_callbacks()
    loggers= create_loggers()

    trainer = create_trainer(
        train_size=train_size,
        train_percentage=train_percentage,
        max_epochs=configs["train_cfg"].max_epochs,
        cfg=configs["train_cfg"],
        loggers=loggers,
        callbacks=callbacks
    )

    return trainer, model, data_module, split_dfs

### Evaluation utils

In [ ]:
def evaluate_from_checkpoint(
    checkpoint_path: str,
    test_ears_df: pd.DataFrame,
    test_wham_df: pd.DataFrame,
    num_samples: int = 3,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
) -> None:
    """Evaluate the MelReGAN model from a checkpoint and visualize results."""
    
    configs = create_configs()
    
    pipeline = create_pipeline(
        audio_preprocessor_cfg=configs["audio_preprocessor_cfg"], 
        audio_augmentor_cfg=configs["audio_augumentor_cfg"], 
        normalizer_cfg=configs["normalizer_cfg"]
    )
    
    generator = MelReGANGenerator(start_filters=configs["train_cfg"].g_filters)
    discriminator = MelReGANDiscriminator(
        in_channels=configs["train_cfg"].d_input_channels, 
        start_filters=configs["train_cfg"].d_filters
    )
    
    scaler = create_scaler(configs["normalizer_cfg"])

    model = MelReGAN.load_from_checkpoint(
        checkpoint_path,
        generator=generator,
        discriminator=discriminator,
        pipeline=pipeline,
        scaler=scaler,
        map_location=device
    )
    
    model.to(device)
    model.eval()

    test_dataset = create_dataset(
        ears_df=test_ears_df,
        wham_df=test_wham_df,
        mixing_audio_cfg=configs["mixing_audio_cfg"],
        train=False,
        skip_ratio=10
    )
    
    hifi_gan = HIFIGAN.from_hparams(
        source="speechbrain/tts-hifigan-libritts-16kHz",
        savedir="tmp_hifigan",
        run_opts={"device": device}
    )
    
    hifi_converter = DBToLogScaler(configs["audio_preprocessor_cfg"]).to(device)

    data_iter = iter(test_dataset)

    for i in range(num_samples):
        try:
            mixed_window, clean_window = next(data_iter)
        except StopIteration:
            print("Reached end of test dataset.")
            break

        mixed_windows = mixed_window.unsqueeze(0).to(device) 
        clean_windows = clean_window.unsqueeze(0).to(device)

        with torch.no_grad():
            noisy_spec_norm, clean_spec_norm = model.pipeline(mixed_windows, clean_windows)

            enhanced_spec_norm = model.generator(noisy_spec_norm)

            noisy_spec_db = model.scaler.denormalize(noisy_spec_norm)
            clean_spec_db = model.scaler.denormalize(clean_spec_norm)
            enhanced_spec_db = model.scaler.denormalize(enhanced_spec_norm)

            noisy_hifi = hifi_converter(noisy_spec_db).squeeze(1)
            clean_hifi = hifi_converter(clean_spec_db).squeeze(1)
            enhanced_hifi = hifi_converter(enhanced_spec_db).squeeze(1)

            audio_noisy = hifi_gan.decode_batch(noisy_hifi).cpu().squeeze().numpy()
            audio_clean = hifi_gan.decode_batch(clean_hifi).cpu().squeeze().numpy()
            audio_enhanced = hifi_gan.decode_batch(enhanced_hifi).cpu().squeeze().numpy()
            
            viz_noisy = noisy_spec_db[0].squeeze().cpu().numpy()
            viz_clean = clean_spec_db[0].squeeze().cpu().numpy()
            viz_enhanced = enhanced_spec_db[0].squeeze().cpu().numpy()

        
        print(f"Sample {i+1}/{num_samples}")
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        kwargs = {'origin': 'lower', 'aspect': 'auto', 'cmap': 'magma', 'vmin': -80, 'vmax': 0}
        
        axes[0].imshow(viz_noisy, **kwargs)
        axes[0].set_title("Input (Noisy)")
        
        axes[1].imshow(viz_enhanced, **kwargs)
        axes[1].set_title("Output (Enhanced)")
        
        axes[2].imshow(viz_clean, **kwargs)
        axes[2].set_title("Target (Clean)")
        plt.show()
        
        print("🔴 Input (Noisy):")
        ipd.display(ipd.Audio(audio_noisy, rate=16000))
        
        print("🟢 Output (Enhanced):")
        ipd.display(ipd.Audio(audio_enhanced, rate=16000))
        
        print("🔵 Target (Clean):")
        ipd.display(ipd.Audio(audio_clean, rate=16000))
        print("\n")




## Train

In [ ]:
trainer, model, data_module, split_dfs = train(train_size=50_000, train_percentage=0.8)
trainer.fit(model, datamodule=data_module)

## Evaluate

In [ ]:
!ls -la /kaggle/working/checkpoints

In [ ]:

# CHECKPOINT_PATH = "checkpoints/mel_bin2bin-epoch=152-val_loss=0.2860.ckpt" # - 15_000 probes
CHECKPOINT_PATH = "checkpoints/mel_bin2bin-epoch=153-val_loss=0.2355.ckpt" # - 50_000 probes


# evaluate_from_checkpoint(
#     checkpoint_path=CHECKPOINT_PATH,
#     test_ears_df=split_dfs["test_ears_df"], # lub val_ears_df jeśli testowy jest pusty
#     test_wham_df=split_dfs["test_wham_df"],
#     num_samples=10
# )